<a href="https://colab.research.google.com/github/kaviyasri2405/machine-learning/blob/main/ex_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Create a sample CSV file for demonstration
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Strong'],
    'Water': ['Warm', 'Warm', 'Warm', 'Cool'],
    'Forecast': ['Same', 'Same', 'Change', 'Change'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}
df_sample = pd.DataFrame(data)
df_sample.to_csv('enjoy_sport.csv', index=False)
print('Sample CSV file (enjoy_sport.csv) created.')

Sample CSV file (enjoy_sport.csv) created.


In [2]:
import numpy as np

def learn_candidate_elimination(training_data):
    # Initialize Specific and General hypotheses
    # The specific hypothesis S starts with the most specific hypothesis (first positive example)
    # The general hypothesis G starts with the most general hypothesis (all '?'s)

    num_attributes = training_data.shape[1] - 1  # Exclude the target column

    # Initialize S to the first positive example
    # Initialize G to a list of lists, each containing ['?', '?', ..., '?']
    S = []
    G = []

    for i, row in training_data.iterrows():
        example = row.iloc[:-1].tolist()  # Features
        target = row.iloc[-1]           # Target concept

        if target == 'Yes':  # Positive example
            if not S:
                S.append(example)
            else:
                # Update S: generalize S if inconsistent with positive example
                # For each hypothesis in S, if it doesn't cover the positive example,
                # generalize it by finding the least general generalization.
                new_S = []
                for s in S:
                    if not _covers(s, example):
                        # Generalize s to cover 'example'
                        generalized_s = _generalize_hypothesis(s, example)
                        new_S.append(generalized_s)
                    else:
                        new_S.append(s)
                S = _remove_redundant_hypotheses(new_S)

            # Remove hypotheses from G that are inconsistent with the positive example
            # i.e., G should not cover a positive example that S does not.
            G = [g for g in G if _covers(g, example)]

        else:  # Negative example
            # For each hypothesis in S, if it incorrectly covers a negative example,
            # specialize it (this part is typically handled by G)
            S = [s for s in S if not _covers(s, example)]

            # For each hypothesis in G, if it incorrectly covers a negative example,
            # specialize it by creating new general hypotheses that are consistent.
            new_G = []
            if not G:
                # Initialize G if it's empty (first example is negative)
                # Add all possible specializations of '?' that don't cover the negative example
                for j in range(num_attributes):
                    temp_g = ['?'] * num_attributes
                    temp_g[j] = example[j]
                    new_G.append(temp_g)
            else:
                for g in G:
                    if _covers(g, example):
                        # Specialize g to not cover 'example'
                        for j in range(num_attributes):
                            if g[j] == '?' and example[j] != '?':
                                temp_g = list(g)
                                temp_g[j] = example[j]
                                new_G.append(temp_g)
                    else:
                        new_G.append(g)
            G = _remove_redundant_hypotheses(new_G)

    # Remove any specific hypothesis that is more general than any general hypothesis
    final_S = []
    for s_hyp in S:
        is_consistent = True
        for g_hyp in G:
            if _is_more_general(s_hyp, g_hyp):
                is_consistent = False
                break
        if is_consistent:
            final_S.append(s_hyp)
    S = final_S

    # Remove any general hypothesis that is more specific than any specific hypothesis
    final_G = []
    for g_hyp in G:
        is_consistent = False
        if not S:
            is_consistent = True # If S is empty, G can contain anything
        else:
            for s_hyp in S:
                if _is_more_general(g_hyp, s_hyp):
                    is_consistent = True
                    break
        if is_consistent:
            final_G.append(g_hyp)
    G = final_G

    return S, G

def _covers(hypothesis, example):
    # Checks if a hypothesis covers an example
    for i in range(len(hypothesis)):
        if hypothesis[i] != '?' and hypothesis[i] != example[i]:
            return False
    return True

def _generalize_hypothesis(hypothesis, example):
    # Generalizes a specific hypothesis to cover an example
    new_hypothesis = list(hypothesis)
    for i in range(len(hypothesis)):
        if new_hypothesis[i] != example[i]:
            new_hypothesis[i] = '?'
    return new_hypothesis

def _is_more_general(h1, h2):
    # Checks if hypothesis h1 is more general than or equal to h2
    # h1 is more general if it covers all examples h2 covers and possibly more.
    # In terms of attribute values, '?' is more general than a specific value.
    for i in range(len(h1)):
        if h1[i] == '?' and h2[i] != '?':
            continue
        elif h1[i] != '?' and h2[i] == '?':
            return False # h1 is more specific than h2 in this attribute
        elif h1[i] != h2[i]:
            return False
    return True

def _remove_redundant_hypotheses(hypotheses):
    # Remove duplicate and more specific hypotheses that are covered by others
    unique_hypotheses = []
    for h1 in hypotheses:
        is_redundant = False
        for h2 in unique_hypotheses:
            if _is_more_general(h2, h1): # If h2 is already more general than h1
                is_redundant = True
                break
            elif _is_more_general(h1, h2): # If h1 is more general than h2, replace h2 with h1
                unique_hypotheses.remove(h2)
                break
        if not is_redundant and h1 not in unique_hypotheses:
            unique_hypotheses.append(h1)
    return unique_hypotheses


In [3]:
# Load the training data from the created CSV file
df_train = pd.read_csv('enjoy_sport.csv')

# Run the Candidate-Elimination algorithm
S, G = learn_candidate_elimination(df_train)

print("Most Specific Hypotheses (S):")
for s_hyp in S:
    print(s_hyp)

print("\nMost General Hypotheses (G):")
for g_hyp in G:
    print(g_hyp)


Most Specific Hypotheses (S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

Most General Hypotheses (G):
['?', '?', '?', 'Strong', '?', '?']
